## **Preparando o ambiente para utilizar o Pyspark**

### Instalando o Java

O Apache Spark depende de outros sistemas, portanto, antes do Spark é preciso instalar as dependências. Primeiro, deve-se instalar o java

----


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

Em seguida, é preciso fazer o download do Spark, e, também, do hadoop, pois o Apache Spark roda sob o HDFS, em sua máquina (no caso aqui, na máquina virtual do Google Colab que você está usando.

In [ ]:
# Fazendo download
# Verificar a versão atual no link: https://spark.apache.org/downloads.html
# Data de Acesso: 01.10.2025
!wget -q https://dlcdn.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz

# Descompactando os arquivos
!tar xf spark-3.5.7-bin-hadoop3.tgz

Pronto! Agora precisamos dizer para o sistema onde encontrar o Java e o Spark, que instalamos a pouco neste ambiente.

In [ ]:
# Importando a biblioteca os
import os

# Definindo a variável de ambiente do Java
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Definindo a variável de ambiente do Spark
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"

A seguir, vamos precisar da biblioteca findspark que vai nos permitir importar pacotes necessários para o funcionamento do pyspark



In [ ]:
# instalando a findspark
!pip install -q findspark

In [ ]:
#importando a findspark
import findspark

# iniciando o findspark
findspark.init()

In [ ]:
# ------------------------------------
# Importanto as bibliotecas
# -----------------
from pyspark.sql import SparkSession
#----------
from pyspark.sql.functions import col, skewness, kurtosis # Para assimetria e Curtose
#----------
import numpy as np # Importanto Numpy
#----------
import matplotlib.pyplot as plt # Importanto Numpy
#----------
import seaborn as sns # Importando Seaborn
# -----------------

In [ ]:
# ------------------------------------
# Iniciando uma sessão do Spark
# ----
# obs:
# appName: Nome do app para a sessão
# -----------------
spark = SparkSession \
        .builder \
        .appName("Tutorial com SPARK - EPBEst") \
        .config("spark.some.config.option", "some-value") \
        .getOrCreate()
# -----------------

Baixando os dados para serem utilizados

In [ ]:
!rm -r base_e-commerce_parquet/

In [ ]:
# ------------------------------------
# Baixando os dados
# ----
#!pip install gdown
!gdown --id 14u4dUuTSYvTOIg4nWPJGai9f8VAXFavy -O base_e-commerce_parquet.zip
# Descompactando os dados
!unzip base_e-commerce_parquet.zip

Criar pasta userdata e fazer upload dos arquivos.

In [ ]:
# ------------------------------------
# Ler dados com extensão PARQUET
# ----
# Localização da base de dados:
# ----
# Número de linhas: 338185
# -----------------
dados = spark.read.parquet("base_e-commerce_parquet/database_compras_parquet.parquet")
# -----------------

In [ ]:
# ------------------------------------
# Verificando a quantidade de linhas
# -----------------
dados.count()
# -----------------

In [ ]:
# ------------------------------------
# Verificando as colunas
# -----------------
dados.printSchema()
# -----------------

In [ ]:
# ------------------------------------
# Mostrando as 20 linhas da variável
# "DsCanalVenda"
# -----------------
dados.select("DsCanalVenda").show()
# -----------------

In [ ]:
dados.show(2,True)

---
## 3 Análise Exploratória - Descritiva
---

In [ ]:
# ------------------------------------
# Selecionando as colunas para análise
# "VrVendaLiquida"
# -----------------
colunas_selecionadas = ["VrVendaLiquida"]
# ------
dados.select(colunas_selecionadas).describe().show()

In [ ]:
# ------------------------------------
# Assimetria e Curtose
# -----------------
colunas_selecionadas = "VrVendaLiquida"
# ------
dados.select(skewness(colunas_selecionadas),kurtosis(colunas_selecionadas)).show()
# ------

### 3.1 Fazendo um Histograma
---

In [ ]:
# ------------------------------------
# Gerando um histograma
# -----------------
colunas_selecionadas = "VrVendaLiquida"
# ------
plot_data = dados.select(colunas_selecionadas).toPandas()
x = plot_data[colunas_selecionadas]
bins = np.arange(0, 94950.000000,4000)
# ------
hist, bin_edges = np.histogram(x,bins,weights=np.zeros_like(x) + 100. / x.size) # make the histogram
# -----
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1)
#-----
# Plot the histogram heights against integers on the x axis
ax.bar(range(len(hist)),hist,width=1,alpha=0.8,ec ='black',color = 'gold')
#-----
# Set the ticks to the middle of the bars
ax.set_xticks([0.5+i for i,j in enumerate(hist)])
#-----
# Set the xticklabels to a string that tells us what the bin edges were
labels =['{}'.format(bins[i+1]) for i,j in enumerate(hist)]
labels.insert(0,'0')
#-----
plt.xlabel(colunas_selecionadas)
plt.ylabel('percentage')
plt.show()

### 3.2 Fazendo um BoxPlot
---

In [ ]:
# ------------------------------------
# Gerando um BoxPlot
# -----------------
colunas_selecionadas = "VrVendaLiquida"
# ------
x = dados.select(colunas_selecionadas).toPandas()
#-----
fig = plt.figure(figsize=(20, 8))
#-----
ax = fig.add_subplot(1, 2, 1)
ax = sns.boxplot(data=x)
ax = fig.add_subplot(1, 2, 2)
ax = sns.violinplot(data=x)

### 3.3 Fazendo um agrupamento e gráfico de barras
---

In [ ]:
# ------------------------------------
# Tabela de Frequencias e
# Gráfico de Barras
# -------
# Funções Necessárias
from pyspark.sql import functions as F
from pyspark.sql.functions import rank, sum, col
from pyspark.sql import Window
# -------
window = Window.rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)
# --
DadosAgrupados = dados.select(['DsCanalVenda', 'VrVendaLiquida']).\
    groupBy('DsCanalVenda').\
    agg(F.count('VrVendaLiquida').alias('VrVendaLiquida_num'),
        F.mean('VrVendaLiquida').alias('VrVendaLiquida_mean'),
        F.min('VrVendaLiquida').alias('VrVendaLiquida_min'),
        F.max('VrVendaLiquida').alias('VrVendaLiquida_max')).\
    withColumn('total', sum(col('VrVendaLiquida_num')).over(window)).\
    withColumn('Percent', col('VrVendaLiquida_num')*100/col('total')).\
    drop(col('total')
    )
# --


In [ ]:
# Visualizando Dados Agrupados
DadosAgrupados.show()


In [ ]:
# ------------------------------------
# Bar Plot - para os dados agrupados
# ----------------
plot_data = DadosAgrupados.toPandas()
# ----------------
labels = plot_data.DsCanalVenda
missing = plot_data.Percent
ind = [x for x, _ in enumerate(labels)]
# ---------
plt.figure(figsize=(10,8))
plt.bar(ind, missing, width=0.8, label='missing', color='gold')

plt.xticks(ind, labels)
plt.ylabel("percentage")
plt.show()


### 3.4 Análise Descritiva Multivariada
---

#### 3.4.1 Correlação


In [ ]:
# -------------------------------------------
from pyspark.mllib.stat import Statistics
import pandas as pd
# -------------
corr_data = dados.select("VrVendaLiquida","VrDescontoTotal")
# -------------
col_names = corr_data.columns
features = corr_data.rdd.map(lambda row: row[0:])
corr_mat = Statistics.corr(features, method="pearson")
corr_df = pd.DataFrame(corr_mat)
corr_df.index, corr_df.columns = col_names, col_names
# -------------
print(corr_df.to_string())

In [ ]:
# -----------------------------
# Scatter Plot
# -------------
import seaborn as sns
# ----------
sns.set(style = "ticks")
# ----------
df = dados.select("VrVendaLiquida","VrDescontoTotal","DsCanalVenda")
df2 = df.toPandas()
sns.pairplot(df2, hue = "DsCanalVenda")
plt.show()
